# Elkinematics Research and Experimentation Playground

This notebook serves as the central interactive sandbox and staging ground for the Elkinematics dysgraphia screening project.

### Workflow Sections:
1. **Environment and Centralized Config** - Load paths and hyperparameters from `scripts/config.py`.
2. **Centralized Model Builder** - Inspect and compile V0 (Baseline CNN), V1 (CNN+XAI), and V2 (Bi-LSTM).
3. **Data Loading and Discovery** - Scan and profile samples in `datasets/` and `val_datasets/`.
4. **Kinematic Feature Extraction** - Extract the 8 standard kinematic features (`delta_x`, `delta_y`, `pressure`, `tiltX`, `tiltY`, `velocity`, `acceleration`, `jerk`) and `latency`.
5. **Global Scaling and Preprocessing** - Normalize data using global Z-score scalers without leakage.
6. **Model Training and Tuning** - Train models with callbacks (EarlyStopping, ReduceLROnPlateau).
7. **Inference and Explainable AI (XAI)** - Predict risk scores and render stroke-level heatmaps.
8. **Evaluation and Performance Metrics** - Accuracy, Recall, Confusion Matrix, and ROC-AUC.
9. **Interactive Sandbox / Scratchpad** - Free-form sandbox for rapid prototyping.

## 1. Environment and Centralized Configuration

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root and scripts directory are accessible
PROJECT_ROOT = Path(os.getcwd()).resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

import config

print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Max Timesteps: {config.MAX_TIMESTEPS} | Features: {config.FEATURES}")
print(f"Active Model V2 Path: {config.MODEL_PATHS['v2']}")
print(f"Scaler Path: {config.SCALER_PATH}")

## 2. Centralized Model Builder and Architecture Summary

In [ ]:
from model_builder import build_model, load_or_build_model

# Build and inspect our flagship V2 Bi-LSTM architecture
model_v2 = build_model(version="v2")
model_v2.summary()

## 3. Data Loading and Discovery

In [ ]:
import glob

train_files = glob.glob(os.path.join(config.DATASET_DIR, "*.csv"))
val_files = glob.glob(os.path.join(config.VAL_DATASET_DIR, "*.csv"))

train_normal = [f for f in train_files if "normal" in os.path.basename(f).lower()]
train_dyslexic = [f for f in train_files if ("dyslexia" in os.path.basename(f).lower() or "dysgraphia" in os.path.basename(f).lower())]

print(f"Training Set:   {len(train_files)} files (Normal: {len(train_normal)}, Dysgraphic/Dyslexic: {len(train_dyslexic)})")
print(f"Validation Set: {len(val_files)} files")

## 4. Kinematic Feature Extraction (Standard 8 Features and Latency)

In [ ]:
from universal_pipeline import extract_universal_features

sample_csv = os.path.join(config.SAMPLE_DIR, "sample_normal.csv")
if os.path.exists(sample_csv):
    sample_df = pd.read_csv(sample_csv)
    features_array = extract_universal_features(sample_df)
    print(f"Raw sample shape: {sample_df.shape}")
    print(f"Extracted feature tensor shape: {features_array.shape} (Timesteps x 9 internal features)")
    print("Feature columns:", config.FEATURE_NAMES + ["dt"])
else:
    print(f"Sample not found at {sample_csv}")

## 5. Global Scaling and Preprocessing Pipeline

In [ ]:
from universal_pipeline import load_and_scale_universal, load_validation_with_train_scalers, get_v2_data

# 1. Load and compute global scalers on training data
print("Processing Training Cohort...")
X_train_seq_scaled, X_train_lat_scaled, y_train = load_and_scale_universal(
    data_dir=config.DATASET_DIR,
    scaler_save_path=config.SCALER_PATH
)

# 2. Transform validation cohort using frozen training scalers (Strict Hold-Out, No Leakage)
print("Processing Validation Cohort with Frozen Scalers...")
X_val_seq_scaled, X_val_lat_scaled, y_val = load_validation_with_train_scalers(
    data_dir=config.VAL_DATASET_DIR,
    scaler_path=config.SCALER_PATH
)

# 3. Format inputs for V2 Multi-Input Model
[X_train_v2_seq, X_train_v2_lat], y_train_td = get_v2_data(X_train_seq_scaled, X_train_lat_scaled, y_train)
[X_val_v2_seq, X_val_v2_lat], y_val_td = get_v2_data(X_val_seq_scaled, X_val_lat_scaled, y_val)

print(f"Scaled Training Sequences: {X_train_v2_seq.shape}")
print(f"Scaled Validation Sequences: {X_val_v2_seq.shape}")

## 6. Model Training and Fine-Tuning

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Build fresh compiled V2 model
experiment_model = build_model("v2", learning_rate=config.LEARNING_RATE)

early_stop = EarlyStopping(monitor='val_loss', patience=config.EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=config.REDUCE_LR_PATIENCE, min_lr=config.MIN_LR, verbose=1)

# Optional: Set to True to train interactive experiment
TRAIN_EXPERIMENT = False

if TRAIN_EXPERIMENT:
    history = experiment_model.fit(
        [X_train_v2_seq, X_train_v2_lat], y_train_td,
        validation_data=([X_val_v2_seq, X_val_v2_lat], y_val_td),
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
else:
    print("Training skipped. Set TRAIN_EXPERIMENT = True to run full training cycle.")

## 7. Inference and Explainable AI (XAI) Heatmap Inspection

In [ ]:
from universal_pipeline import unified_predict

# Load production trained model
prod_model = load_or_build_model("v2", filepath=config.MODEL_PATHS["v2"])

test_sample_path = os.path.join(config.SAMPLE_DIR, "sample_dyslexia.csv")
if os.path.exists(test_sample_path):
    test_df = pd.read_csv(test_sample_path)
    score, heatmap = unified_predict(test_df, prod_model, version="v2", scaler_path=config.SCALER_PATH)
    
    print(f"Sample: {os.path.basename(test_sample_path)}")
    print(f"Risk Score: {score:.4f} -> {'Atypical / Dysgraphic' if score > 0.5 else 'Typical / Normal'}")
    
    # Plot XAI Risk Heatmap across stroke timesteps
    plt.figure(figsize=(10, 3))
    plt.plot(heatmap[:len(test_df)], color='red', label='Per-Timestep Risk')
    plt.axhline(0.5, color='gray', linestyle='--', label='Threshold (0.5)')
    plt.title("Explainable AI (XAI) Error Probability Heatmap")
    plt.xlabel("Stroke Timestep (ms)")
    plt.ylabel("Anomaly Risk")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Test sample not found at {test_sample_path}")

## 8. Evaluation and Metrics

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

# Generate predictions on validation set
val_preds_td = prod_model.predict([X_val_v2_seq, X_val_v2_lat], verbose=0)
# Aggregate time-distributed output to sequence-level score
val_scores = np.mean(val_preds_td[:, :, 0], axis=1)
val_binary_preds = (val_scores > config.CLASSIFICATION_THRESHOLD).astype(int)

acc = accuracy_score(y_val, val_binary_preds)
rec = recall_score(y_val, val_binary_preds)
prec = precision_score(y_val, val_binary_preds)
f1 = f1_score(y_val, val_binary_preds)
auc = roc_auc_score(y_val, val_scores)

print("=" * 40)
print("MODEL V2 VALIDATION METRICS")
print("=" * 40)
print(f"Accuracy : {acc * 100:.2f}%")
print(f"Recall   : {rec * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"F1-Score : {f1 * 100:.2f}%")
print(f"ROC-AUC  : {auc:.4f}")
print("=" * 40)

## 9. Interactive Sandbox / Scratchpad

Use this cell to experiment freely with new kinematic calculations, new layer tweaks, or exploratory tests before committing to `scripts/`.

In [ ]:
# --- SANDBOX SCRATCHPAD ---
print("Sandbox ready for experimentation.")
